In [25]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from typing import Sequence, Any

# 1. Define the wrapper module with the custom VJP
class AdjointHook(nn.Module):
    """
    A wrapper module that applies a custom VJP to any given layer
    to capture its incoming gradient (adjoint) during the backward pass.
    """
    layer: nn.Module  # The layer to wrap (e.g., nn.Dense)

    @nn.compact
    def __call__(self, *args, **kwargs):
        # Define the function to which we'll attach the custom VJP
        @jax.custom_vjp
        def layer_with_hook(params, *args, **kwargs):
            return self.layer.apply(params, *args, **kwargs)

        # --- Define the Forward and Backward Passes ---

        def layer_fwd(params, *args, **kwargs):
            # Execute the original layer's forward pass
            output = self.layer.apply(params, *args, **kwargs)
            # Return output and residuals for the backward pass
            return output, (params, args, kwargs)

        def layer_bwd(res, g):
            # res contains residuals: (params, args, kwargs)
            # g is the incoming gradient (the adjoint we want to capture)
            params, args, kwargs = res

            print(f"--- Captured Adjoint for layer: {self.layer.name} ---")
            print(g)
            print(res)
            print("-" * 30)

            # Calculate the VJP of the original wrapped layer
            # This computes the gradients w.r.t. params and inputs
            _, vjp_fun = jax.vjp(lambda p, *a, **kw: self.layer.apply(p, *a, **kw), params, *args, **kwargs)
            
            # The VJP function returns a tuple of gradients
            grad_params, *grad_args = vjp_fun(g)
            
            # The custom VJP's backward pass must return a tuple of gradients
            # matching the inputs of the forward pass (params, *args, **kwargs).
            # JAX handles the kwargs gradients automatically.
            return (grad_params,) + tuple(grad_args)

        # Attach the custom forward and backward functions
        layer_with_hook.defvjp(layer_fwd, layer_bwd)
        
        # Get the parameters for the wrapped layer
        layer_params = self.param('wrapped_layer', self.layer.init, *args, **kwargs)

        return layer_with_hook(layer_params, *args, **kwargs)


class residual_dense(nn.Module):
    @nn.compact 
    def __call__(self, x):
        x1 = nn.Dense(3)(x)
        return x1 + x
# class residual_dense(nn.Module):
#     def setup(self):
#         self.A = self.param(
#             'A', 
#             lambda rng: jnp.array([[1.0, -2.0, -1.0], 
#                                    [-2.0, 1.0, -2.0], 
#                                    [-1.0, -2.0, 1.0]]) / 4
#         )
#         self.b = self.param(
#             'b', 
#             lambda rng: jnp.array([0.0, 0.0, 0.0])
#         )
#     @nn.compact
#     def __call__(self, x):
#         # Compute Ax + b
#         x1 = jnp.dot(x, self.A) + self.b
#         return x1 + x  # Residual connection
    
# 2. Define the main model using the 'setup' method
class MLP(nn.Module):
    """A simple multi-layer perceptron."""
    feature_dims: Sequence[int]

    def setup(self):
        # Create a list of layers. We wrap each one with our AdjointHook.
        self.layers = [
            # AdjointHook(nn.Dense(features=dim, name=f"dense_{i}"))
            AdjointHook(residual_dense(name=f"dense_{i}"))
            for i, dim in enumerate(self.feature_dims)
        ]
        # self.activation = nn.relu

    def __call__(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            # Apply activation function after all but the last layer
            # if i < len(self.layers) - 1:
            #     x = self.activation(x)
        return x

# --- Example Usage ---

# Setup the model and data
key = jax.random.PRNGKey(0)
x_input = jnp.ones((1, 3))  # Input shape: (batch, features)

# Define a 3-layer MLP. This can be extended easily.
# Input(10) -> Dense(64) -> ReLU -> Dense(32) -> ReLU -> Dense(4)
layer_sizes = [3, 3, 3]
model = MLP(feature_dims=layer_sizes)

# Initialize parameters
params = model.init(key, x_input)

print(params)

# Define the loss function to trigger gradient calculation
def loss_fn(p):
    output = model.apply(p, x_input)
    return jnp.sum(output)

# Calculate gradients. This will execute the `layer_bwd` functions.
print("Starting gradient calculation...\n")
grads = jax.grad(loss_fn)(params)
print("\nGradient calculation complete.")

{'params': {'layers_0': {'wrapped_layer': {'params': {'Dense_0': {'kernel': Array([[ 0.7810132 , -0.6700104 , -0.36642563],
       [-0.8254334 ,  0.2224843 , -0.13227795],
       [-0.07376321,  0.43821934,  0.12365075]], dtype=float32), 'bias': Array([0., 0., 0.], dtype=float32)}}}}, 'layers_1': {'wrapped_layer': {'params': {'Dense_0': {'kernel': Array([[-0.65958077,  1.3005502 ,  0.22552262],
       [-0.5636002 ,  0.25953552, -0.066361  ],
       [ 0.47236112, -0.81795067,  0.42846018]], dtype=float32), 'bias': Array([0., 0., 0.], dtype=float32)}}}}, 'layers_2': {'wrapped_layer': {'params': {'Dense_0': {'kernel': Array([[-1.0870695 , -0.23957756,  0.0404082 ],
       [ 0.7779552 ,  0.8072604 , -0.73189443],
       [ 0.19455186, -0.5123879 ,  0.10224909]], dtype=float32), 'bias': Array([0., 0., 0.], dtype=float32)}}}}}}
Starting gradient calculation...

--- Captured Adjoint for layer: dense_2 ---
[[1. 1. 1.]]
({'params': {'Dense_0': {'bias': Array([0., 0., 0.], dtype=float32), 'kernel'

In [32]:
# A = jnp.array([[1.0, -2.0, -1.0], 
#                                    [-2.0, 1.0, -2.0], 
#                                    [-1.0, -2.0, 1.0]]) / 4

A = params['params']['layers_2']['wrapped_layer']['params']['Dense_0']['kernel']
A
A @ jnp.ones((3, 1)) + jnp.ones((3, 1))

Array([[-0.2862388],
       [ 1.8533211],
       [ 0.7844131]], dtype=float32)